In [ ]:
# ============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ============================================================================

import os
import json
import time
import copy
import zipfile
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from tqdm import tqdm


# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Paths ──────────────────────────────────────────────────────────────────
# Dataset path – relative to this notebook's location.
# Expected folder structure:
#
#   <repo-root>/
#     dental_caries_dataset/
#       final_dataset/
#         train/
#         val/
#         test/
#     training_results/          <- created automatically
#     densenet121-dental-caries.ipynb
#
# To use a different location simply change DATA_DIR below.
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                        "dental_caries_dataset", "final_dataset")

# Output directory – all weights, metrics, and plots land here.
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                          "training_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
NUM_EPOCHS    = 30
BATCH_SIZE    = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-3
PATIENCE      = 7          # Early stopping patience
NUM_WORKERS   = 0          # Use 0 on Windows to avoid DataLoader issues
IMG_SIZE      = 224
CLASS_NAMES   = ['caries', 'healthy']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Data:   {DATA_DIR}')
print(f'Output: {OUTPUT_DIR}')


In [ ]:
# ============================================================================
# CELL 2: DATASET SETUP
# ============================================================================

# ── Transforms ─────────────────────────────────────────────────────────────
# Images are already 224x224 RGB after preprocessing. We apply standard
# ImageNet normalization + training augmentations.

"""" train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])"""
''''
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # Zoom in slightly
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),                # X-rays can often be flipped vertically
    transforms.RandomRotation(degrees=30),               # Increased from 10 to 30
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Slightly stronger
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


eval_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Datasets & DataLoaders ─────────────────────────────────────────────────
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, 'train'), transform=train_transforms
)
val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, 'val'), transform=eval_transforms
)
test_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, 'test'), transform=eval_transforms
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)'''
# ============================================================================
# CELL 2: DATASET & DATALOADERS (UPDATED)
# ============================================================================
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler

print(f'{"="*70}')
print(f'  DATA PREPARATION (IMBALANCE FIXED)')
print(f'{"="*70}')

# 1. Gentle Data Augmentation for Medical Images
train_transforms = transforms.Compose([
    # Resize slightly larger, then take a random 224x224 crop (simulates slight camera shifting)
    transforms.Resize((236, 236)),
    transforms.RandomCrop(224),
    
    # Left/Right flips are fine for X-rays, but NEVER Vertical Flips (upside-down anatomy)
    transforms.RandomHorizontalFlip(p=0.5),
    
    # Very slight rotation (patients might tilt their head slightly, but not 30 degrees)
    transforms.RandomRotation(degrees=10),
    
    # Subtle lighting changes (different X-ray machines have slightly different exposures)
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load Datasets (FIXED to use DATA_DIR)
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'), transform=val_test_transforms)
test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), transform=val_test_transforms)

CLASS_NAMES = train_dataset.classes
print(f'Classes: {CLASS_NAMES}')

# 2. Implement WeightedRandomSampler to balance batches
class_counts = [0] * len(CLASS_NAMES)
for _, label in train_dataset.samples:
    class_counts[label] += 1
    
print(f'Training class counts: {class_counts}')

# Calculate weights (inverse frequency)
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[label] for _, label in train_dataset.samples]

# Create the sampler (Draws samples randomly based on their weights)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# 3. Create DataLoaders (CRITICAL: shuffle=False when using a sampler)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders ready! Training batches are now perfectly balanced.')

# Print class mapping and counts
print(f'\nClass mapping: {train_dataset.class_to_idx}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

# Count per class
train_targets = np.array(train_dataset.targets)
n_caries  = (train_targets == train_dataset.class_to_idx['caries']).sum()
n_healthy = (train_targets == train_dataset.class_to_idx['healthy']).sum()
print(f'Train distribution — caries: {n_caries}, healthy: {n_healthy}')

# ── Class weights for imbalanced data ──────────────────────────────────────
# ImageFolder sorts alphabetically: caries=0, healthy=1
# We use pos_weight for BCEWithLogitsLoss: weight for the POSITIVE class.
# Since caries >> healthy, and caries=0, healthy=1:
#   pos_weight = n_negative / n_positive = n_caries / n_healthy
pos_weight = torch.tensor([n_caries / n_healthy], dtype=torch.float32).to(DEVICE)
print(f'pos_weight (for healthy class): {pos_weight.item():.3f}')




In [ ]:

# ============================================================================
# CELL 3: MODEL BUILDING (UPDATED)
# ============================================================================
import torch.nn as nn
from torchvision import models
from torchvision.models import DenseNet121_Weights

print(f'\n{"="*70}')
print(f'  BUILDING MODEL & OPTIMIZER')
print(f'{"="*70}')

def build_model():
    model = models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    
    # Freeze early layers, unfreeze the last block
    for param in model.parameters():
        param.requires_grad = False
    for param in model.features.denseblock4.parameters():
        param.requires_grad = True
    for param in model.features.norm5.parameters():
        param.requires_grad = True
        
    # STRONGER regularization: Dropout increased to 0.5
    num_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5), 
        nn.Linear(num_features, 1)
    )
    return model

model = build_model().to(DEVICE)

# STRONGER Weight Decay: Increased to 1e-3
WEIGHT_DECAY = 1e-3 
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), 
                             lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# REMOVED pos_weight because the Sampler now balances the data automatically
criterion = nn.BCEWithLogitsLoss() 

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

print("DenseNet121 built with 50% Dropout.")
print("Optimizer and Scheduler initialized.")

In [ ]:
# ============================================================================
# CELL 4: TRAINING LOOP
# ============================================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch. Returns average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='  Train', leave=False,
                bar_format='{l_bar}{bar:30}{r_bar}')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate. Returns average loss, accuracy, all predictions and labels."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds  = []
    all_labels = []

    pbar = tqdm(loader, desc='  Val  ', leave=False,
                bar_format='{l_bar}{bar:30}{r_bar}')
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels_t = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels_t)

            running_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels_t).sum().item()
            total += labels_t.size(0)

            all_preds.extend(preds.cpu().numpy().flatten().astype(int))
            all_labels.extend(labels.numpy())

            pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels


# ── Training ───────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [],   'val_acc': [],
    'lr': []
}

best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0

print(f'\n{"="*70}')
print(f'  TRAINING — {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}')
print(f'{"="*70}')

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]['lr']

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )

    # Validate
    val_loss, val_acc, _, _ = validate(
        model, val_loader, criterion, DEVICE
    )

    # Scheduler step
    scheduler.step(val_loss)

    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    elapsed = time.time() - epoch_start

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} │ '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} │ '
          f'LR: {current_lr:.2e} │ {elapsed:.1f}s')

    # Early stopping & checkpointing
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f'         ↳ New best model saved (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\n  Early stopping triggered at epoch {epoch} '
                  f'(no improvement for {PATIENCE} epochs)')
            break

# Load best weights
model.load_state_dict(best_model_wts)
print(f'\nLoaded best model weights (val_loss={best_val_loss:.4f})')

# Save best weights
weights_path = os.path.join(OUTPUT_DIR, 'densenet121_caries_best.pth')
torch.save(best_model_wts, weights_path)
print(f'Saved weights to: {weights_path}')

# ── Plot Training Curves (UPDATED: LOSS ONLY) ──────────────────────────────
# We use a single subplot now instead of (1, 2)
fig, ax = plt.subplots(figsize=(8, 6))

# Plot only Loss
ax.plot(history['train_loss'], label='Train Loss', linewidth=2, color='#1f77b4', marker='o', markersize=4)
ax.plot(history['val_loss'], label='Val Loss', linewidth=2, color='#ff7f0e', marker='x', markersize=4)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training & Validation Loss Progress', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Change the filename to reflect that it is only the loss curve
curves_path = os.path.join(OUTPUT_DIR, 'training_loss_curve.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.close() 

print(f'Saved training loss curve to: {curves_path}')


# ADD THIS TO SEE THE PLOT IN THE NOTEBOOK:
from IPython.display import Image, display
display(Image(filename=curves_path))



In [ ]:
# ============================================================================
# CELL 5A: TEST SET EVALUATION
# ============================================================================
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print(f'\n{"="*70}')
print(f'  TEST SET EVALUATION')
print(f'{"="*70}')

# Force all layers in the model to disable in-place operations globally
for module in model.modules():
    if hasattr(module, 'inplace'):
        module.inplace = False

# Evaluate
test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion, DEVICE)

print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

print(f'\n{"-"*50}')
print('Classification Report:')
print(f'{"-"*50}')
report = classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4)
print(report)

# --- NEW: PRINT RAW CONFUSION MATRIX ---
cm = confusion_matrix(test_labels, test_preds)
print(f'\n{"-"*50}')
print('Confusion Matrix (Raw):')
print(f'{"-"*50}')
print(cm)
print(f'{"-"*50}\n')

# Calculate and save metrics
acc = accuracy_score(test_labels, test_preds)
prec = precision_score(test_labels, test_preds, average='weighted', zero_division=0)
rec = recall_score(test_labels, test_preds, average='weighted', zero_division=0)
f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=0)

metrics = {
    'test_loss': round(test_loss, 4),
    'accuracy': round(acc, 4),
    'precision_weighted': round(prec, 4),
    'recall_weighted': round(rec, 4),
    'f1_weighted': round(f1, 4),
    'class_names': CLASS_NAMES
}

metrics_path = os.path.join(OUTPUT_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved metrics to: {metrics_path}')

# Confusion Matrix Heatmap Image
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, 
            annot_kws={'size': 16}, ax=ax)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved confusion matrix heatmap to: {cm_path}')

# ============================================================================
# CELL: VISUALIZE TRAINING HISTORY
# ============================================================================
import matplotlib.pyplot as plt

# Set style for a cleaner look
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# 1. Plot Loss
ax1.plot(history['train_loss'], label='Train Loss', color='#1f77b4', lw=2, marker='o', ms=4)
ax1.plot(history['val_loss'], label='Val Loss', color='#ff7f0e', lw=2, marker='x', ms=4)
ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Loss Value', fontsize=12)
ax1.legend()

# 2. Plot Accuracy
ax2.plot(history['train_acc'], label='Train Acc', color='#2ca02c', lw=2, marker='o', ms=4)
ax2.plot(history['val_acc'], label='Val Acc', color='#d62728', lw=2, marker='x', ms=4)
ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()
from IPython.display import Image, display

print("--- CONFUSION MATRIX ---")
display(Image(filename=cm_path))


In [ ]:
# ============================================================================
# CELL 5B: GRAD-CAM IMPLEMENTATION (FIXED)
# ============================================================================
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# --- THE CRITICAL DENSENET FIX ---
# We must override the hardcoded inplace=True in DenseNet's forward pass
def densenet_forward_no_inplace(self, x):
    features = self.features(x)
    out = F.relu(features, inplace=False) # <--- Forces inplace=False
    out = F.adaptive_avg_pool2d(out, (1, 1))
    out = torch.flatten(out, 1)
    out = self.classifier(out)
    return out

# Apply the override to our model instance
model.forward = densenet_forward_no_inplace.__get__(model)
# ---------------------------------

class GradCAM:
    """
    Grad-CAM for DenseNet-121.
    Targets the final feature extraction layer (features.norm5).
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        self.target_layer.register_forward_hook(self._forward_hook)
        self.target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output.clone().detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].clone().detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()

        # Temporarily suppress inplace operations for standard layers
        for m in self.model.modules():
            if hasattr(m, 'inplace'):
                m.inplace = False

        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = (torch.sigmoid(output) >= 0.5).int().item()

        self.model.zero_grad()

        if class_idx == 1:
            output.backward(retain_graph=True) 
        else:
            (-output).backward(retain_graph=True)

        # Pool gradients across spatial dims
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().numpy()

        # Resize to input size (Assuming IMG_SIZE is 224)
        cam = cv2.resize(cam, (224, 224))
        
        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

def apply_gradcam_overlay(image_tensor, heatmap, alpha=0.4):
    """Overlay Grad-CAM heatmap on original image."""
    # THE FIX: Added .detach() before .cpu().numpy()
    img = image_tensor.detach().cpu().numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = std * img + mean
    img = np.clip(img, 0, 1)

    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = np.float32(heatmap_colored) / 255
    heatmap_colored = heatmap_colored[:, :, ::-1] # Convert BGR to RGB

    overlay = (1 - alpha) * img + alpha * heatmap_colored
    overlay = np.clip(overlay, 0, 1)
    return img, overlay


print(f'\n{"="*70}')
print(f'  GENERATING GRAD-CAM FOR TEST IMAGE')
print(f'{"="*70}')

# Initialize GradCAM
gradcam = GradCAM(model, model.features.norm5)

# Grab a single batch to test the visualization
inputs, labels = next(iter(test_loader))
inputs = inputs.to(DEVICE)
inputs.requires_grad_(True)

# Generate Heatmap for the first image in the batch
heatmap = gradcam.generate(inputs[0:1], class_idx=labels[0].item())
original_img, overlay_img = apply_gradcam_overlay(inputs[0], heatmap)

# Plot Result
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(original_img)
ax1.set_title(f'Actual Class: {CLASS_NAMES[labels[0].item()]}')
ax1.axis('off')
ax2.imshow(overlay_img)
ax2.set_title('DenseNet121 Grad-CAM Focus')
ax2.axis('off')
plt.show()

In [ ]:
# ============================================================================
# CELL 6: ZIP TRAINING RESULTS
# ============================================================================
# After running this cell you will find training_results.zip next to the
# notebook.  Copy that file into the repo root and extract it so that:
#
#   training_results/
#     densenet121_caries_best.pth
#     metrics.json
#     training_history.json
#     training_loss_curve.png
#     confusion_matrix.png
#
# Those files are required by app.py (Streamlit dashboard).
# ============================================================================
import os
import json
import zipfile

print(f'\n {"=" * 70}')
print(f'  PACKAGING RESULTS')
print(f' {"=" * 70}')

# 1. Save training history into the output folder first
history_path = os.path.join(OUTPUT_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

# 2. Zip the entire output directory
zip_path = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                        'training_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname   = os.path.relpath(file_path, OUTPUT_DIR)
            zf.write(file_path, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'\nResults zipped successfully!')
print(f'  Location : {zip_path}')
print(f'  Size     : {zip_size_mb:.2f} MB')
print(f'\n {"=" * 70}')
print(f'  CONTENTS OF ZIP:')
print(f' {"=" * 70}')
print(f'  densenet121_caries_best.pth  — Model weights')
print(f'  metrics.json                 — Accuracy, Precision, Recall, F1')
print(f'  training_history.json        — Per-epoch loss/acc history')
print(f'  training_loss_curve.png      — Loss vs Epoch plot')
print(f'  confusion_matrix.png         — Confusion matrix heatmap')
print(f'\n  Extract training_results.zip into the repo root to run app.py.')
